# ML Capstone — Refresh Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring

**Research question:** Can safe aggregated search/content signals rank existing pages for refresh review more effectively than the Week-4 rule baseline?

This notebook is the reproducible companion to the deployed paper. It intentionally reports association and decision-support findings, not causal Google-ranking claims.

## 1. Question and decision

The supported decision is reviewer prioritization: which existing content should be inspected first when editorial capacity is limited? The output is a ranked queue, not an automatic publishing decision.

In [1]:
lane = 'Refresh / Content Opportunity Scoring'
decision = 'prioritize manual review of existing content'
print(f'Lane: {lane}')
print(f'Decision: {decision}')

Lane: Refresh / Content Opportunity Scoring
Decision: prioritize manual review of existing content



## 2. Data

The bundled anonymized content-refresh slice contains 30,000 eligible rows after filtering to positive 90-day impressions, content age of at least 90 days, and unique content IDs. The public paper excludes client names, domains, URLs, private queries, credentials, and raw exports.

In [2]:
import json
meta = json.load(open('data/processed/feature_metadata.json'))
print(f"Eligible rows: {meta['prepared_rows']:,}")
print(f"Declining-label rows: {meta['declining_rows']:,}")
print(f"Declining-label rate: {meta['declining_rate']:.3f}")

Eligible rows: 30,000
Declining-label rows: 16,262
Declining-label rate: 0.542


## 3. Methodology and leakage audit

Target: `trend_direction == 'down'`. Predictors include demand, visibility, content length, age/freshness, CTR, average position, engagement, scroll behavior, and safe categorical tiers. The comparison uses the same client-holdout split for the baseline and models. Target-derived trend fields are excluded from the model feature list.

Because predictors and the label summarize the same observed window, this is not a clean prospective forecasting test. That limitation is central to the interpretation.

In [3]:
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
features = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
print('Validation split: client_holdout')
print('Target-derived trend features in model list:', int(any(x in features for x in ['trend_pct','trend_direction'])))
print('Models: logistic_regression, decision_tree, random_forest')
print('Selection metric: Precision@50')

Validation split: client_holdout
Target-derived trend features in model list: 0
Models: logistic_regression, decision_tree, random_forest
Selection metric: Precision@50


## 4. Results — model vs baseline

The random forest is selected by Precision@50. All reported methods were evaluated on the same holdout rows.

In [4]:
results = [('Rules baseline',.627,.468,.240),('Logistic regression',.700,.522,.400),('Decision tree',.742,.575,.540),('Random forest',.750,.618,.740)]
for name, auc, ap, p50 in results:
    print(f'{name} | ROC AUC {auc:.3f} | Avg precision {ap:.3f} | P@50 {p50:.3f}')

Rules baseline | ROC AUC 0.627 | Avg precision 0.468 | P@50 0.240
Logistic regression | ROC AUC 0.700 | Avg precision 0.522 | P@50 0.400
Decision tree | ROC AUC 0.742 | Avg precision 0.575 | P@50 0.540
Random forest | ROC AUC 0.750 | Avg precision 0.618 | P@50 0.740


## 5. Ranked recommendations

1. Review the highest-score visible pages first.
2. Check CTR/snippet opportunities before broad rewrites when low CTR is a reason code.
3. Distinguish engagement problems from demand problems.
4. Use content age and visibility for triage, not as automatic rewrite triggers.
5. Keep the rules baseline as an explainability and sanity-check fallback.
6. Upgrade the next experiment to a future-window label so period T features predict a period T+1 outcome.

In [5]:
summary = {'top_queue':10,'high_confidence':3605,'medium_confidence':11395,'low_confidence':15000}
for k,v in summary.items(): print(f'{k.replace('_',' ').title()}: {v:,}' if isinstance(v,int) else f'{k}: {v}')
print('Recommended first action: manual review of high-score visible pages')

Top queue: 10 anonymized rows
High-confidence items: 3,605
Medium-confidence items: 11,395
Low-confidence items: 15,000
Recommended first action: manual review of high-score visible pages


## 6. Limitations and claim audit

This work measures associations in one anonymized release. It does not prove Google's ranking algorithm, causal refresh impact, or a guaranteed traffic lift. The cross-sectional label and shared observation window can inflate apparent predictive strength for prospective use. Results should therefore be treated as directional and decision-support evidence.

## 7. Reproducibility

Run `python scripts/run_all.py` to regenerate the model outputs from the bundled anonymized slice. Charts are in `outputs/charts/`; the generated report is `outputs/model_report.md`; this notebook and the earlier track notebooks are under `work/notebooks/`.

## 8. Public paper

The deployed paper is served from GitHub Pages at the URL stored in `submission/paper_url.txt`.

## 9. ML-12 closing

**5-minute demo:** question → data contract → baseline → model comparison → ranked queue → limitations.

**Social cut:** I built a public-safe content refresh ranking model on anonymized search data, compared it against rules, and turned the result into a reviewer-first action queue.

**Employer summary:** I framed a real ML decision, controlled leakage at the feature-list level, compared models against a fixed baseline on the same grouped split, and shipped the findings as a reproducible research paper.